In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import OptimConfig
from vizopt.templates.euler.stars_vs_circles import (
    optimize_radially_convex_sets_and_circles_from_graph,
)
from vizopt.examples.sets import make_animals_graph, make_multiples_of_primes_graph
from vizopt.schedules import make_term_schedules

# Sets of multiples

In [ ]:
primes = [2, 3, 5]
G = make_multiples_of_primes_graph(primes)
elements = sorted(n for n in G.nodes if G.out_degree(n) == 0)

In [ ]:
n_iters = 8000
best_params = {
    "collision_delay": 0.27,
    "collision_ramp":  0.33,
    "exclusion_delay": 0.32,
    "exclusion_ramp":  0.47,
    "area_delay":      0.31,
    "area_ramp":       0.27,
    "perimeter_delay": 0.69,
    "perimeter_ramp":  0.45,
    "attraction_peak": 0.69,
    "attraction_ramp": 0.18,
}
term_schedules = make_term_schedules(best_params, n_iters)

In [ ]:
snapshot_cb = SnapshotCallback(every=50)

named_results, named_circles_out, history, problem = optimize_radially_convex_sets_and_circles_from_graph(
    G,
    weight_area=1.0,
    weight_perimeter=1.0,
    weight_exclusion=10.0,
    weight_smoothness=2.0,
    weight_position_anchor=0.3,
    weight_circle_collision=20.0,
    weight_set_attraction=5.0,
    term_schedules=term_schedules,
    optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    callback=snapshot_cb,
)
print(f"Captured {len(snapshot_cb.snapshots)} snapshots")

In [ ]:
set_colors = ["tomato", "steelblue", "mediumorchid", "mediumseagreen"]
set_node_names = [f"multiples_of_{p}" for p in primes]
set_labels = [f"Multiples of {p}" for p in primes]

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color, label in zip(set_node_names, set_colors, set_labels):
    result = named_results[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2, label=label)
    ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

for n in elements:
    r_orig = G.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_out[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, str(n), ha="center", va="center", fontsize=10, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.legend(loc="upper right", fontsize=11)
ax.set_title("Multiples of 2, 3, 5 among 1-11\nStar-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()

In [ ]:
pd.DataFrame(history).set_index("iteration").plot(marker=".", figsize=(7, 3))
plt.ylabel("Loss value")
plt.title("Optimization history")
plt.tight_layout()

In [ ]:
svg = snapshots_to_animated_svg(problem, snapshot_cb.snapshots, fps=5, size=600, history=history)
SVG(data=svg)

# Animals taxonomy

In [ ]:
G_a = make_animals_graph()
elements_a = sorted(n for n in G_a.nodes if G_a.out_degree(n) == 0)
set_names_a = [n for n in nx.topological_sort(G_a) if G_a.out_degree(n) > 0]

In [ ]:
named_results_a, named_circles_a, history_a, problem_a = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G_a,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.01,
        weight_circle_collision=100.0,
        weight_set_attraction=0.1,
        circle_collision_alpha=1.0,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

In [ ]:
history_a[-1]

In [ ]:
def boundary_label_position(center, radii, angles, member_circles, inset=1.01):
    """Boundary point with maximum clearance from member circles, pulled slightly inward."""
    center = np.asarray(center)
    bx = center[0] + radii * np.cos(angles)
    by = center[1] + radii * np.sin(angles)
    bp = np.stack([bx, by], axis=1)  # (K, 2)

    member_circles = np.asarray(member_circles)
    if member_circles.ndim == 2 and len(member_circles) > 0:
        dists = np.linalg.norm(bp[:, None, :] - member_circles[None, :, :2], axis=2)
        clearances = (dists - member_circles[None, :, 2]).min(axis=1)
    else:
        clearances = radii

    k = int(np.argmax(clearances))
    return center + inset * radii[k] * np.array([np.cos(angles[k]), np.sin(angles[k])])

In [ ]:
leaf_set_a = {n for n in G_a.nodes if G_a.out_degree(n) == 0}
colors_a = plt.cm.tab10(np.linspace(0, 0.9, len(set_names_a)))

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color in zip(set_names_a, colors_a):
    result = named_results_a[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2)
    #ax.plot(cx, cy, "+", color=color, markersize=10, markeredgewidth=2)

    members = [n for n in nx.descendants(G_a, set_name) if n in leaf_set_a]
    member_circles = np.array([named_circles_a[n] for n in members])
    lx, ly = boundary_label_position(np.array([cx, cy]), radii, angles, member_circles)
    ax.text(lx, ly, set_name, ha="center", va="center", fontsize=9, fontweight="bold", color=color)

for n in elements_a:
    r_orig = G_a.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_a[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, n, ha="center", va="center", fontsize=9, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.set_title("Animal taxonomy — star-shaped boundaries with movable element circles")
ax.axis("off")
plt.tight_layout()

# Label rectangles

Re-run the animals example with `label_rect_size` to reserve a floating label area at the top of each set boundary.

In [ ]:
named_results_l, named_circles_l, history_l, problem_l = (
    optimize_radially_convex_sets_and_circles_from_graph(
        G_a,
        weight_area=1.0,
        weight_perimeter=1.0,
        weight_exclusion=10.0,
        weight_smoothness=2.0,
        weight_position_anchor=0.01,
        weight_circle_collision=100.0,
        weight_set_attraction=0.1,
        circle_collision_alpha=1.0,
        label_rect_size=(0.6, 0.15),
        weight_label_enclosure=20.0,
        weight_label_element_exclusion=20.0,
        weight_label_collision=10.0,
        weight_label_top=1.0,
        term_schedules=term_schedules,
        optim_config=OptimConfig(n_iters=n_iters, learning_rate=0.002),
    )
)

In [ ]:
from matplotlib.patches import FancyBboxPatch

colors_l = plt.cm.tab10(np.linspace(0, 0.9, len(set_names_a)))
plot_label_box = False

fig, ax = plt.subplots(figsize=(9, 9))

for set_name, color in zip(set_names_a, colors_l):
    result = named_results_l[set_name]
    cx, cy = result["center"]
    radii = result["radii"]
    angles = result["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.15, color=color)
    ax.plot(bx, by, color=color, linewidth=2)

    lx, ly = result["label_center"]
    hw, hh = 0.6, 0.15
    if plot_label_box:
        ax.add_patch(FancyBboxPatch(
            (lx - hw, ly - hh), 2 * hw, 2 * hh,
            boxstyle="round,pad=0.05", linewidth=1.2,
            edgecolor=color, facecolor=color, alpha=0.25,
        ))
    ax.text(lx, ly, set_name, ha="center", va="center",
            fontsize=8, fontweight="bold", color=color)

for n in elements_a:
    r_orig = G_a.nodes[n]["r"]
    cx_out, cy_out, _ = named_circles_l[n]
    ax.add_patch(plt.Circle((cx_out, cy_out), r_orig, facecolor="lightyellow", alpha=0.9,
                            edgecolor="dimgray", linewidth=1.5))
    ax.text(cx_out, cy_out, n, ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.15)
ax.set_title("Animal taxonomy — label rectangles reserved at top of each set boundary")
ax.axis("off")
plt.tight_layout()

# Illustrating objective terms: area, perimeter, smoothness, convexity

Five circles are arranged in a C-shape (opening to the right). We run four separate
optimisations, each focusing on one term (weight = 20) while the others stay low (weight = 2).
Circle positions are strongly anchored so that only the boundary shape changes.

| Term | Effect |
|------|--------|
| **Area** | Minimises enclosed area → tight boundary, may pull inward where there are no circles |
| **Perimeter** | Minimises boundary length → prefers round/circular shapes |
| **Smoothness** | Penalises abrupt radius changes between adjacent angles → boundary follows circles but without spikes |
| **Convexity** | Penalises concave turns → explicitly forces a convex boundary |

In [ ]:
# 5 circles on a C-arc (opening to the right): angles 60°, 120°, 180°, 240°, 300°
arc_angles_c = np.array([np.pi / 3, 2 * np.pi / 3, np.pi, 4 * np.pi / 3, 5 * np.pi / 3])
R_arc = 2.5

G_terms = nx.DiGraph()
for i, a in enumerate(arc_angles_c):
    G_terms.add_node(f"c{i}", center=[float(R_arc * np.cos(a)), float(R_arc * np.sin(a))], r=0.4)
G_terms.add_node("S")
for i in range(5):
    G_terms.add_edge("S", f"c{i}")

leaf_names_terms = [n for n in G_terms.nodes if G_terms.out_degree(n) == 0]

In [ ]:
_kw = dict(
    weight_exclusion=0.0,
    weight_position_anchor=50.0,
    weight_circle_collision=0.0,
    weight_set_attraction=0.0,
    circle_collision_alpha=0.0,
    optim_config=OptimConfig(n_iters=3000, learning_rate=0.003),
)

_cases_def = [
    ("Area",       "steelblue",      dict(weight_area=10.0, weight_perimeter=1.0,  weight_smoothness=0.5, weight_convexity=0.0)),
    ("Perimeter",  "tomato",         dict(weight_area=1.0,  weight_perimeter=10.0, weight_smoothness=0.5, weight_convexity=0.0)),
    ("Smoothness", "mediumpurple",   dict(weight_area=1.0,  weight_perimeter=1.0,  weight_smoothness=10.0, weight_convexity=0.0)),
    ("Convexity",  "mediumseagreen", dict(weight_area=1.0,  weight_perimeter=1.0,  weight_smoothness=0.5, weight_convexity=100.0, convexity_alpha=1.0)),
]

cases = []
for _name, _color, _params in _cases_def:
    _res, _, _, _ = optimize_radially_convex_sets_and_circles_from_graph(G_terms, **_kw, **_params)
    cases.append((_name, _color, _params, _res))

In [ ]:
_ABBREV = {
    "weight_area": "area", "weight_perimeter": "perim",
    "weight_smoothness": "smooth", "weight_convexity": "conv",
    "convexity_alpha": "α",
}

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, (name, color, params, named_res) in zip(axes, cases):
    parts = [f"{_ABBREV[k]}={v:g}" for k, v in params.items() if k in _ABBREV]
    title = f"{name}\n({', '.join(parts)})"

    res = named_res["S"]
    cx, cy = res["center"]
    radii = res["radii"]
    angles = res["angles"]
    bx = np.append(cx + radii * np.cos(angles), cx + radii[0] * np.cos(angles[0]))
    by = np.append(cy + radii * np.sin(angles), cy + radii[0] * np.sin(angles[0]))
    ax.fill(bx, by, alpha=0.2, color=color)
    ax.plot(bx, by, color=color, linewidth=2)

    for n in leaf_names_terms:
        ccx, ccy = G_terms.nodes[n]["center"]
        r = G_terms.nodes[n]["r"]
        ax.add_patch(plt.Circle((ccx, ccy), r, facecolor="lightyellow", alpha=0.9,
                                edgecolor="dimgray", linewidth=1.5))

    ax.set_aspect("equal")
    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-4.5, 4.5)
    ax.set_title(title, fontsize=10, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle("Boundary shapes with different dominant objective terms\n(5 circles in a C-shape, positions anchored)", fontsize=11)
plt.tight_layout()

## Convexity term inspection: morphing from convex to concave

A square (W = 1.5) is deformed in two directions from a baseline:

- **t > 0** — lerp toward a circle (genuinely convex at every t)
- **t = 0** — baseline square
- **t < 0** — 4-fold harmonic perturbation `r_rect + t × 0.4W × cos(4θ)`: sides indent, corners protrude → concave

Two penalty variants are shown:
- **α = 0** (pure quadratic) `sum(max(0, −sin α)²)` — near-zero gradient for small concavities
- **α = 1** (quadratic + linear) `sum(max(0, −sin α)² + max(0, −sin α))` — constant gradient even for mild concavities

In [ ]:
import jax.numpy as jnp
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec

from vizopt.components.stars import (
    _multi_term_area,
    _multi_term_convexity,
    _multi_term_perimeter,
    _multi_term_smoothness,
)

K = 64
W, H = 1.5, 1.5
angles_k = np.linspace(0, 2 * np.pi, K, endpoint=False).astype(np.float32)

# Square baseline radii: r(θ) = min(W/|cosθ|, H/|sinθ|)
r_rect = np.minimum(
    W / np.abs(np.cos(angles_k)).clip(1e-9),
    H / np.abs(np.sin(angles_k)).clip(1e-9),
).astype(np.float32)

# Circle radius for the convex direction
r_circle = float(np.mean(r_rect))

# 4-fold perturbation for the concave direction (cos 4θ peaks at mid-sides)
perturb = (0.4 * W * np.cos(4 * angles_k)).astype(np.float32)


def radii_at(t: float) -> np.ndarray:
    if t >= 0:
        return (1.0 - t) * r_rect + t * r_circle
    else:
        return np.maximum(r_rect + t * perturb, 0.02)


def losses_at(t: float) -> dict:
    r = radii_at(t)
    ov = {"centers": jnp.zeros((1, 2)), "radii": jnp.array(r[None, :])}
    ip0 = {"angles": jnp.array(angles_k), "convexity_alpha": jnp.float32(0.0)}
    ip1 = {"angles": jnp.array(angles_k), "convexity_alpha": jnp.float32(1.0)}
    return {
        "convexity_q":  float(_multi_term_convexity(ov, ip0)),
        "convexity_ql": float(_multi_term_convexity(ov, ip1)),
        "smoothness":   float(_multi_term_smoothness(ov, ip0)),
        "area":         float(_multi_term_area(ov, ip0)),
        "perimeter":    float(_multi_term_perimeter(ov, ip0)),
    }


t_dense = np.linspace(-1.0, 1.0, 400)
t_show  = np.linspace(-1.0, 1.0, 7)
all_losses = [losses_at(t) for t in t_dense]

# ── figure ────────────────────────────────────────────────────────────────────
n = len(t_show)
fig = plt.figure(figsize=(16, 8))
outer = GridSpec(2, 1, figure=fig, height_ratios=[1.2, 1.2], hspace=0.4)
gs_top = GridSpecFromSubplotSpec(1, n, subplot_spec=outer[0], wspace=0.05)
gs_bot = GridSpecFromSubplotSpec(1, 2, subplot_spec=outer[1], wspace=0.35)

# shape thumbnails
for i, t in enumerate(t_show):
    ax = fig.add_subplot(gs_top[0, i])
    r = radii_at(t)
    bx = np.append(r * np.cos(angles_k), r[0] * np.cos(angles_k[0]))
    by = np.append(r * np.sin(angles_k), r[0] * np.sin(angles_k[0]))
    color = plt.cm.RdYlGn((t + 1) / 2)
    ax.fill(bx, by, alpha=0.3, color=color)
    ax.plot(bx, by, color=color, linewidth=1.5)
    ax.set_xlim(-2.8, 2.8)
    ax.set_ylim(-2.0, 2.0)
    ax.set_aspect("equal")
    ax.set_title(f"t = {t:+.2f}", fontsize=9)
    ax.axis("off")

# convexity: quadratic vs quadratic+linear
ax_c = fig.add_subplot(gs_bot[0, 0])
ax_c.plot(t_dense, [l["convexity_q"]  for l in all_losses],
          color="mediumseagreen", linewidth=2, label="α = 0  (quadratic)")
ax_c.plot(t_dense, [l["convexity_ql"] for l in all_losses],
          color="mediumseagreen", linewidth=2, linestyle="--", label="α = 1  (quadratic + linear)")
ax_c.axvspan(-1.0, 0.0, alpha=0.07, color="red")
ax_c.axvspan(0.0,  1.0, alpha=0.07, color="green")
ax_c.axvline(0, color="black", linewidth=0.8, alpha=0.4)
for t in t_show:
    ax_c.axvline(t, color="gray", linestyle="--", linewidth=0.7, alpha=0.4)
ax_c.set_xlabel("t")
ax_c.set_ylabel("Loss value")
ax_c.set_title("Convexity term")
ax_c.legend(fontsize=9)

# other terms for comparison
ax_o = fig.add_subplot(gs_bot[0, 1])
for name, col in [("smoothness", "mediumpurple"), ("area", "steelblue"), ("perimeter", "tomato")]:
    ax_o.plot(t_dense, [l[name] for l in all_losses], label=name, color=col, linewidth=2)
ax_o.axvspan(-1.0, 0.0, alpha=0.07, color="red")
ax_o.axvspan(0.0,  1.0, alpha=0.07, color="green")
ax_o.axvline(0, color="black", linewidth=0.8, alpha=0.4)
for t in t_show:
    ax_o.axvline(t, color="gray", linestyle="--", linewidth=0.7, alpha=0.4)
ax_o.set_xlabel("t")
ax_o.legend()
ax_o.set_title("Other terms")

plt.suptitle(
    "Objective terms along the shape morph\nconcave (t < 0)  ←——  square (t = 0)  ——→  circle (t > 0)",
    fontsize=11,
)
plt.tight_layout()